<a href="https://colab.research.google.com/github/laveena-majeed/Video_game_analysis/blob/main/Video_game_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Python Module – Data Preparation & Exploratory Data Analysis (EDA)

## Objective
This module focuses on preparing raw video game datasets for analysis and visualization by performing:
- Data loading
- Data cleaning
- Missing value handling using business logic
- Genre normalization and validation
- Dataset merging for SQL and Power BI usage

The output of this module is a **clean, structured dataset** suitable for analytical and business intelligence tasks.


## 1.1 Load Required Libraries

Essential Python libraries are imported for data processing, visualization, and analysis.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast

pd.set_option('display.max_columns', None)

## 1.2 Load Datasets

The raw datasets are loaded using Pandas.  
Two datasets are used:
- `games.csv` → Contains game metadata and engagement metrics  
- `vgsales.csv` → Contains regional and global sales data


In [ ]:
games_df = pd.read_csv('/content/games.csv')
vgsales_df = pd.read_csv('/content/vgsales.csv')

print("Games Dataset Shape:", games_df.shape)
print("VG Sales Dataset Shape:", vgsales_df.shape)

games_df.head(), vgsales_df.head()

Games Dataset Shape: (1512, 14)
VG Sales Dataset Shape: (16598, 11)


(   Unnamed: 0                                    Title  Release Date  \
 0           0                               Elden Ring  Feb 25, 2022   
 1           1                                    Hades  Dec 10, 2019   
 2           2  The Legend of Zelda: Breath of the Wild  Mar 03, 2017   
 3           3                                Undertale  Sep 15, 2015   
 4           4                            Hollow Knight  Feb 24, 2017   
 
                                                 Team  Rating Times Listed  \
 0     ['Bandai Namco Entertainment', 'FromSoftware']     4.5         3.9K   
 1                               ['Supergiant Games']     4.3         2.9K   
 2  ['Nintendo', 'Nintendo EPD Production Group No...     4.4         4.3K   
 3                                 ['tobyfox', '8-4']     4.2         3.5K   
 4                                    ['Team Cherry']     4.4           3K   
 
   Number of Reviews                                             Genres  \
 0             

## 1.3 Initial Data Understanding

This step helps understand:
- Data types
- Null values
- Statistical distribution
- Overall dataset structure


In [ ]:
games_df.info()
vgsales_df.info()

games_df.describe()
vgsales_df.describe()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1512 entries, 0 to 1511
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         1512 non-null   int64  
 1   Title              1512 non-null   object 
 2   Release Date       1512 non-null   object 
 3   Team               1511 non-null   object 
 4   Rating             1499 non-null   float64
 5   Times Listed       1512 non-null   object 
 6   Number of Reviews  1512 non-null   object 
 7   Genres             1512 non-null   object 
 8   Summary            1511 non-null   object 
 9   Reviews            1512 non-null   object 
 10  Plays              1512 non-null   object 
 11  Playing            1512 non-null   object 
 12  Backlogs           1512 non-null   object 
 13  Wishlist           1512 non-null   object 
dtypes: float64(1), int64(1), object(12)
memory usage: 165.5+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16598 entr

,Rank,Year,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
count,16598.000000,16327.000000,16598.000000,16598.000000,16598.000000,16598.000000,16598.000000
mean,8300.605254,2006.406443,0.264667,0.146652,0.077782,0.048063,0.537441
std,4791.853933,5.828981,0.816683,0.505351,0.309291,0.188588,1.555028
min,1.000000,1980.000000,0.000000,0.000000,0.000000,0.000000,0.010000
25%,4151.250000,2003.000000,0.000000,0.000000,0.000000,0.000000,0.060000
50%,8300.500000,2007.000000,0.080000,0.020000,0.000000,0.010000,0.170000
75%,12449.750000,2010.000000,0.240000,0.110000,0.040000,0.040000,0.470000
max,16600.000000,2020.000000,41.490000,29.020000,10.220000,10.570000,82.740000


## 1.4 Duplicate Values – Analysis & Business Logic

Duplicate values can distort analytical results. However, business logic must be applied before removing records.

### Business Rules:
- The same game on different platforms is **NOT a duplicate**.
- Only **exact duplicate rows** must be removed.


In [ ]:
games_df.duplicated().sum()
vgsales_df.duplicated().sum()


np.int64(0)

In [ ]:
games_df.duplicated(subset=['Title']).sum()

vgsales_df.duplicated(subset=['Name', 'Platform', 'Year']).sum()


np.int64(2)

In [ ]:
print(games_df.columns)
print(vgsales_df.columns)


Index(['Unnamed: 0', 'Title', 'Release Date', 'Team', 'Rating', 'Times Listed',
       'Number of Reviews', 'Genres', 'Summary', 'Reviews', 'Plays', 'Playing',
       'Backlogs', 'Wishlist'],
      dtype='object')
Index(['Rank', 'Name', 'Platform', 'Year', 'Genre', 'Publisher', 'NA_Sales',
       'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales'],
      dtype='object')


### Action Taken
- Same game released on different platforms is considered **valid**.
- Only **exact duplicate rows** are removed.


### 1.5 Missing Values – Business Logic & Handling

In [ ]:
# Fill missing ratings with median
games_df['Rating'] = games_df['Rating'].fillna(games_df['Rating'].median())

# Engagement metrics filled with zero
cols = ['Plays', 'Wishlist', 'Backlogs']
for col in cols:
    games_df[col] = games_df[col].fillna(0)

# Regional sales filled with zero
sales_cols = ['NA_Sales', 'EU_Sales', 'JP_Sales', 'Other_Sales', 'Global_Sales']
for col in sales_cols:
    vgsales_df[col] = vgsales_df[col].fillna(0)

### 1.6 Date Format Standardization

In [ ]:
games_df['Release Date'] = pd.to_datetime(games_df['Release Date'], errors='coerce')
games_df['Release_Year'] = games_df['Release Date'].dt.year

### 1.7 Genre Cleaning – Invalid Entries Handling (Corrected)

In [ ]:
def safe_literal_eval(x):
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        return [] # Return an empty list for malformed strings

games_df['Genres'] = games_df['Genres'].apply(safe_literal_eval)
print(f"Shape after ast.literal_eval: {games_df.shape}")

games_df = games_df.explode('Genres')
print(f"Shape after explode: {games_df.shape}")

games_df['Genres'] = games_df['Genres'].str.strip().str.title()
print(f"Shape after strip and title: {games_df.shape}")

# Filter out potential empty strings or None values that might result from previous steps
games_df = games_df[games_df['Genres'] != '']

print("Unique genres after explode and title casing:", games_df['Genres'].unique())

valid_genres = [
    'Action', 'Adventure', 'Arcade', 'Casual', 'Family', 'Fighting', 'Indie',
    'Misc', 'Party', 'Platform', 'Puzzle', 'Racing', 'Rpg', 'Shooter',
    'Simulation', 'Sports', 'Strategy', 'Visual Novel', 'Turn Based Strategy'
]
games_df = games_df[games_df['Genres'].isin(valid_genres)]

print("Shape of games_df after genre cleaning:", games_df.shape)

Shape after ast.literal_eval: (1512, 15)
Shape after explode: (3645, 15)
Shape after strip and title: (3645, 15)
Unique genres after explode and title casing: ['Adventure' 'Rpg' 'Brawler' 'Indie' 'Turn Based Strategy' 'Platform'
 'Simulator' 'Strategy' 'Puzzle' 'Shooter' 'Music' 'Fighting' 'Arcade'
 'Visual Novel' 'Card & Board Game' 'Tactical' 'Racing' 'Point-And-Click'
 'Moba' 'Sport' 'Real Time Strategy' 'Quiz/Trivia' nan 'Pinball']
Shape of games_df after genre cleaning: (3178, 15)


### 1.8 Merge Sales & Engagement Data

In [ ]:
merged_df = pd.merge(
    games_df,
    vgsales_df,
    left_on='Title',
    right_on='Name',
    how='inner'
)

print("Shape of merged_df after merging:", merged_df.shape)
merged_df.head()

Shape of merged_df after merging: (2432, 26)


,Unnamed: 0,Title,Release Date,Team,Rating,Times Listed,Number of Reviews,Genres,Summary,Reviews,Plays,Playing,Backlogs,Wishlist,Release_Year,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales
0,5,Minecraft,2011-11-18,['Mojang Studios'],4.3,2.3K,2.3K,Adventure,Minecraft focuses on allowing the player to ex...,['Minecraft is what you make of it. Unfortunat...,33K,1.8K,1.1K,230,2011.0,73,Minecraft,X360,2013.0,Misc,Microsoft Game Studios,5.58,2.83,0.02,0.77,9.20
1,5,Minecraft,2011-11-18,['Mojang Studios'],4.3,2.3K,2.3K,Adventure,Minecraft focuses on allowing the player to ex...,['Minecraft is what you make of it. Unfortunat...,33K,1.8K,1.1K,230,2011.0,169,Minecraft,PS3,2014.0,Misc,Sony Computer Entertainment,1.97,2.51,0.00,0.94,5.42
2,5,Minecraft,2011-11-18,['Mojang Studios'],4.3,2.3K,2.3K,Adventure,Minecraft focuses on allowing the player to ex...,['Minecraft is what you make of it. Unfortunat...,33K,1.8K,1.1K,230,2011.0,298,Minecraft,PS4,2014.0,Misc,Sony Computer Entertainment Europe,1.38,1.87,0.12,0.65,4.02
3,5,Minecraft,2011-11-18,['Mojang Studios'],4.3,2.3K,2.3K,Adventure,Minecraft focuses on allowing the player to ex...,['Minecraft is what you make of it. Unfortunat...,33K,1.8K,1.1K,230,2011.0,644,Minecraft,XOne,2014.0,Misc,Microsoft Game Studios,1.43,0.76,0.00,0.22,2.41
4,5,Minecraft,2011-11-18,['Mojang Studios'],4.3,2.3K,2.3K,Adventure,Minecraft focuses on allowing the player to ex...,['Minecraft is what you make of it. Unfortunat...,33K,1.8K,1.1K,230,2011.0,715,Minecraft,PSV,2014.0,Misc,Sony Computer Entertainment Europe,0.28,0.79,0.87,0.32,2.25


### Feature Engineering

In [ ]:
merged_df['Engagement_Score'] = (
    merged_df['Plays'] + merged_df['Playing'] +
    merged_df['Backlogs'] + merged_df['Wishlist']
)

merged_df['Sales_Category'] = pd.cut(
    merged_df['Global_Sales'],
    bins=[0, 0.5, 2, 5, 10, 100],
    labels=['Low', 'Medium', 'High', 'Very High', 'Blockbuster']
)

merged_df['Rating_Category'] = pd.cut(
    merged_df['Rating'],
    bins=[0, 4, 6, 8, 10],
    labels=['Poor', 'Average', 'Good', 'Excellent']
)

### Dataset Optimization for BI Tools

In [ ]:
final_df = merged_df.drop(columns=['Summary', 'Reviews'])

print("Shape of final_df:", final_df.shape)
final_df.head()

Shape of final_df: (2432, 27)


,Unnamed: 0,Title,Release Date,Team,Rating,Times Listed,Number of Reviews,Genres,Plays,Playing,Backlogs,Wishlist,Release_Year,Rank,Name,Platform,Year,Genre,Publisher,NA_Sales,EU_Sales,JP_Sales,Other_Sales,Global_Sales,Engagement_Score,Sales_Category,Rating_Category
0,5,Minecraft,2011-11-18,['Mojang Studios'],4.3,2.3K,2.3K,Adventure,33K,1.8K,1.1K,230,2011.0,73,Minecraft,X360,2013.0,Misc,Microsoft Game Studios,5.58,2.83,0.02,0.77,9.20,33K1.8K1.1K230,Very High,Average
1,5,Minecraft,2011-11-18,['Mojang Studios'],4.3,2.3K,2.3K,Adventure,33K,1.8K,1.1K,230,2011.0,169,Minecraft,PS3,2014.0,Misc,Sony Computer Entertainment,1.97,2.51,0.00,0.94,5.42,33K1.8K1.1K230,Very High,Average
2,5,Minecraft,2011-11-18,['Mojang Studios'],4.3,2.3K,2.3K,Adventure,33K,1.8K,1.1K,230,2011.0,298,Minecraft,PS4,2014.0,Misc,Sony Computer Entertainment Europe,1.38,1.87,0.12,0.65,4.02,33K1.8K1.1K230,High,Average
3,5,Minecraft,2011-11-18,['Mojang Studios'],4.3,2.3K,2.3K,Adventure,33K,1.8K,1.1K,230,2011.0,644,Minecraft,XOne,2014.0,Misc,Microsoft Game Studios,1.43,0.76,0.00,0.22,2.41,33K1.8K1.1K230,High,Average
4,5,Minecraft,2011-11-18,['Mojang Studios'],4.3,2.3K,2.3K,Adventure,33K,1.8K,1.1K,230,2011.0,715,Minecraft,PSV,2014.0,Misc,Sony Computer Entertainment Europe,0.28,0.79,0.87,0.32,2.25,33K1.8K1.1K230,High,Average


### Exporting Final Clean Dataset

In [ ]:
final_df.to_csv("final_video_game_analytics_dataset.csv", index=False)

In [ ]:
from google.colab import files
files.download("final_video_game_analytics_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Final DataFrame Shapes (Confirmation)

In [ ]:
print(games_df.shape)
print(vgsales_df.shape)
print(merged_df.shape)
print(final_df.shape)

(3178, 15)
(16598, 11)
(2432, 29)
(2432, 27)
